## Goal

This notebook compares several known trading-model families on the same historical candles:

- moving-average momentum;
- warmup absolute momentum hold;
- EMA trend following;
- Donchian breakout;
- Bollinger + RSI mean reversion;
- MACD + RSI momentum;
- ATR volatility breakout;
- walk-forward random forest;
- walk-forward histogram gradient boosting.

Every model implements the same `SignalModel` interface and emits `long_entry_signal` / `long_exit_signal` columns.

## 1. Setup

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display


def find_project_root(start: Path) -> Path:
    """Find the project root by walking up from a start directory.

    :param start: directory used as the search starting point
    :return: nearest parent directory containing ``pyproject.toml``
    """

    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    msg = "Could not find project root with pyproject.toml"
    raise FileNotFoundError(msg)


PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / "src"
FREQTRADE_DIR = PROJECT_ROOT / "trading" / "freqtrade"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

PROJECT_ROOT

WindowsPath('C:/WorkPrograms/TimeAnalysis')

## 2. Experiment configuration

The default period is intentionally long enough for a first serious comparison. If the ML cells are too slow, set `INCLUDE_ML = False` for a fast rule-based pass. Set `RUN_DOWNLOAD = True` when you want to refresh local candles from the public Bybit API.

In [2]:
from time_analysis.backtesting import (
    SignalBacktestConfig,
    compare_signal_models_across_pairs,
    load_freqtrade_ohlcv,
)
from time_analysis.data_sources import download_and_save_bybit_ohlcv
from time_analysis.models import default_model_zoo

PAIRS = ["BTC/USDT", "ETH/USDT"]
TIMEFRAME = "1h"
START_DATE = "2024-01-01"
END_DATE = "2026-06-01"
RUN_DOWNLOAD = False
INCLUDE_ML = True
ALLOW_SHORT = False

backtest_config = SignalBacktestConfig(
    initial_balance=1000.0,
    fee_rate=0.001,
    periods_per_year=365 * 24,
    allow_short=ALLOW_SHORT,
)

models = default_model_zoo(include_ml=INCLUDE_ML)
[model.name for model in models]

['BuyAndHoldBenchmarkModel',
 'WarmupMomentumHoldModel',
 'SmaMomentumModel',
 'EmaTrendModel',
 'DonchianBreakoutModel',
 'BollingerRsiMeanReversionModel',
 'MacdRsiTrendModel',
 'AtrVolatilityBreakoutModel',
 'RandomForestDirectionModel',
 'HistGradientBoostingDirectionModel']

## 3. Download and load candles

In [3]:
if RUN_DOWNLOAD:
    for pair in PAIRS:
        path = download_and_save_bybit_ohlcv(
            pair=pair,
            timeframe=TIMEFRAME,
            start=START_DATE,
            end=END_DATE,
            freqtrade_dir=FREQTRADE_DIR,
        )
        print("saved", path)

candles_by_pair = {
    pair: load_freqtrade_ohlcv(FREQTRADE_DIR, pair, TIMEFRAME)
    for pair in PAIRS
}

for pair, candles in candles_by_pair.items():
    print(pair, candles.index.min(), candles.index.max(), len(candles))

BTC/USDT 2024-01-01 00:00:00+00:00 2026-06-01 00:00:00+00:00 21169
ETH/USDT 2024-01-01 00:00:00+00:00 2026-06-01 00:00:00+00:00 21169


## 4. Compare models

In [4]:
pair_metrics, aggregate_metrics, pair_results = compare_signal_models_across_pairs(
    candles_by_pair=candles_by_pair,
    models=models,
    config=backtest_config,
)

display(aggregate_metrics)
display(pair_metrics.sort_values(["pair", "total_return"], ascending=[True, False]))

C:\WorkPrograms\TimeAnalysis\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\WorkPrograms\TimeAnalysis\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


,model,average_return,minimum_return,maximum_return,positive_pairs,non_negative_pairs,worst_drawdown,average_sharpe,total_trades,average_win_rate,average_profit_factor
0,WarmupMomentumHoldModel,0.360790,0.000000,0.721580,1,2,-0.500851,0.354353,1,0.500000,inf
1,BuyAndHoldBenchmarkModel,0.303195,-0.125833,0.732224,1,1,-0.652847,0.482917,2,0.500000,inf
2,DonchianBreakoutModel,-0.048323,-0.091560,-0.005086,0,0,-0.611296,0.092884,280,0.324460,1.040542
3,SmaMomentumModel,-0.159037,-0.172649,-0.145425,0,0,-0.543081,-0.002234,552,0.346283,1.008238
4,AtrVolatilityBreakoutModel,-0.242705,-0.382432,-0.102977,0,0,-0.535725,-0.352074,661,0.307732,0.928980
5,EmaTrendModel,-0.304243,-0.389436,-0.219050,0,0,-0.639274,-0.236411,529,0.279831,0.945493
6,MacdRsiTrendModel,-0.305540,-0.568848,-0.042231,0,0,-0.608537,-0.680597,945,0.297512,0.886385
7,BollingerRsiMeanReversionModel,-0.477974,-0.578311,-0.377637,0,0,-0.650580,-0.792367,397,0.553959,0.734424
8,RandomForestDirectionModel,-0.491655,-0.544021,-0.439288,0,0,-0.695103,-0.544836,1227,0.553737,0.879546
9,HistGradientBoostingDirectionModel,-0.849681,-0.900241,-0.799120,0,0,-0.925040,-1.765640,2110,0.472376,0.726644


,pair,model,final_equity,total_return,annual_return,max_drawdown,sharpe_ratio,total_trades,win_rate,profit_factor,exposure
0,BTC/USDT,BuyAndHoldBenchmarkModel,1732.223540,0.732224,0.255271,-0.500851,0.713207,1,1.000000,inf,0.999953
1,BTC/USDT,WarmupMomentumHoldModel,1721.579864,0.721580,0.252073,-0.500851,0.708707,1,1.000000,inf,0.996599
2,BTC/USDT,DonchianBreakoutModel,994.914335,-0.005086,-0.002108,-0.366039,0.129231,143,0.349650,1.044825,0.354386
3,BTC/USDT,SmaMomentumModel,827.350701,-0.172649,-0.075432,-0.543081,-0.077735,269,0.356877,0.983765,0.527564
4,BTC/USDT,BollingerRsiMeanReversionModel,622.363370,-0.377637,-0.178187,-0.426385,-0.712286,197,0.527919,0.745446,0.155699
5,BTC/USDT,AtrVolatilityBreakoutModel,617.568085,-0.382432,-0.180814,-0.535725,-0.708570,342,0.333333,0.840933,0.249516
6,BTC/USDT,EmaTrendModel,610.564022,-0.389436,-0.184671,-0.639274,-0.456129,265,0.249057,0.882974,0.528036
7,BTC/USDT,RandomForestDirectionModel,560.712255,-0.439288,-0.212908,-0.541314,-0.580539,666,0.521021,0.869542,0.376919
8,BTC/USDT,MacdRsiTrendModel,431.151565,-0.568848,-0.293999,-0.608537,-1.452152,477,0.280922,0.746715,0.221125
9,BTC/USDT,HistGradientBoostingDirectionModel,200.880066,-0.799120,-0.485309,-0.823400,-1.750446,1089,0.461892,0.737045,0.462421


## 5. Inspect the best candidate

In [5]:
best_row = aggregate_metrics.iloc[0]
best_model_name = best_row["model"]
best_pair = pair_metrics.loc[
    pair_metrics["model"].eq(best_model_name),
].sort_values("total_return", ascending=False).iloc[0]["pair"]
best_result = pair_results[best_pair][best_model_name]

print("Best aggregate model:", best_model_name)
print("Best pair:", best_pair)
display(pd.Series(best_result.metrics, name="value").to_frame())
display(best_result.trades.tail(20))

Best aggregate model: WarmupMomentumHoldModel
Best pair: BTC/USDT


,value
model,WarmupMomentumHoldModel
final_equity,1721.579864
total_return,0.72158
annual_return,0.252073
max_drawdown,-0.500851
sharpe_ratio,0.708707
total_trades,1
win_rate,1.0
profit_factor,inf
exposure,0.996599


,side,entry_time,exit_time,entry_price,exit_price,gross_return,net_return
0,long,2024-01-04 00:00:00+00:00,2026-06-01 00:00:00+00:00,42874.0,73884.9,0.723303,0.721303


## 6. Equity and drawdown

In [6]:
equity = best_result.equity_curve

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=equity.index,
        y=equity["equity"],
        mode="lines",
        name="Equity",
    )
)
fig.update_layout(
    title=f"{best_pair} / {best_model_name}: equity",
    xaxis_title="Time",
    yaxis_title="Account value",
    template="plotly_white",
)
fig.show()

drawdown_fig = go.Figure()
drawdown_fig.add_trace(
    go.Scatter(
        x=equity.index,
        y=equity["drawdown"] * 100,
        mode="lines",
        fill="tozeroy",
        name="Drawdown",
    )
)
drawdown_fig.update_layout(
    title=f"{best_pair} / {best_model_name}: drawdown",
    xaxis_title="Time",
    yaxis_title="Drawdown, %",
    template="plotly_white",
)
drawdown_fig.show()

## 7. Promotion rule

A model is a promotion candidate only when it has positive total return, acceptable drawdown, enough trades, and a stable-looking equity curve. The next step is to wrap the selected model in a thin Freqtrade strategy adapter and run a full Freqtrade backtest.